<a href="https://colab.research.google.com/github/esetafinau-gif/AAI2025/blob/2026fall/SelfReflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GEMINI_API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
import google.generativeai as genai

# Initialize the Gemini API with a stable model name identified from the list of available models
gemini_model = genai.GenerativeModel('gemini-pro-latest')

In [6]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [7]:
# Function to generate an initial summary
def generate_summary(model, text):
    response = model.generate_content(f"Summarize the following text for a general audience in approximately 50 words: {text}")
    return response.text

In [8]:
# Function to generate the self-reflection prompt
def generate_self_reflection_prompt(summary, original_text, requirements):
    prompt = f"""
    You are an expert editor and AI critic tasked with rigorously evaluating and improving a summary.

    Here is the original source text:
    ---
    {original_text}
    ---

    Here is the initial summary to evaluate:
    ---
    {summary}
    ---

    Here are the core requirements and rubric you must evaluate against:
    {requirements}

    Please conduct a structured self-reflection assessment. Fill out the critique card below by stating whether each requirement is fully "MET" or "NOT MET", providing a detailed explanation of your judgment with evidence from both texts, and detailing a concrete improvement action.

    Your output must adhere strictly to this format:

    **Critique:**
    - **Accuracy**: [MET / NOT MET] - [Detailed explanation of any omitted core points or added misinformation]
    - **Clarity**: [MET / NOT MET] - [Analysis of sentence structure complexity, jargon used, or readability issues]
    - **Audience**: [MET / NOT MET] - [Analysis of whether a layperson can easily understand the text]
    - **Length**: [MET / NOT MET] - [Exact word count of initial summary and comparison to the ~50-word constraint]
    - **Formatting**: [MET / NOT MET] - [Is it a single coherent paragraph?]

    **Revision Plan:**
    [Briefly list the specific adjustments you will make in the revised summary to fix the identified gaps]

    **Revised Summary:**
    [Your optimized, high-quality, revised summary addressing all critiques above]
    """
    return prompt

In [9]:
# Function to perform the self-reflection and extract the revised summary
def perform_self_reflection(model, long_text, reflection_requirements):
    # 1. Generate initial summary
    initial_summary = generate_summary(model, long_text)
    print("Initial Summary:")
    print(initial_summary)
    print("-" * 30)

    # 2. Generate self-reflection prompt
    self_reflection_prompt = generate_self_reflection_prompt(initial_summary, long_text, reflection_requirements)
    print("Self-Reflection Prompt Generated.")

    # 3. Get AI's critique and revised summary
    critique_response = model.generate_content(self_reflection_prompt)
    critique_text = critique_response.text
    print("\nAI Critique and Revised Summary:")
    print(critique_text)
    print("-" * 30)

    # 4. Extract the revised summary (simple extraction for demonstration)
    revised_summary_start_tag = "**Revised Summary:**"
    if revised_summary_start_tag in critique_text:
        revised_summary = critique_text.split(revised_summary_start_tag)[1].strip()
    else:
        revised_summary = "Could not extract revised summary."

    print("\nExtracted Revised Summary:")
    print(revised_summary)

    return initial_summary, revised_summary, critique_text

In [10]:
# Define the sample text and evaluation rubric requirements
long_text = """
Photosynthesis is a multi-step biological process used by plants, algae, and certain bacteria to harness energy from sunlight and convert it into chemical energy. This chemical energy is stored in carbohydrate molecules, such as sugars, which are synthesized from carbon dioxide and water. The process initiates when light energy is absorbed by proteins called reaction centers that contain green chlorophyll pigments. In plants, these proteins are held inside organelles called chloroplasts, which are most abundant in leaf cells.
In addition to sustaining plant life, photosynthesis is critical for global ecology: it releases oxygen into the atmosphere as a byproduct, which is vital for the survival of aerobic organisms. Furthermore, it acts as a primary sink for atmospheric carbon dioxide, playing a pivotal role in regulating Earth's climate. Without this process, life as we know it would cease to exist.
"""

reflection_requirements = """
- Accuracy: Must include light energy absorption, conversion to chemical energy (sugars), and oxygen release.
- Clarity: Clear structure without overly dense academic jargon.
- Audience: Tailored to a general audience (explain scientific terms if used).
- Length: Approximately 50 words.
- Formatting: Must be formatted as a single cohesive paragraph.
"""

In [11]:
import google.api_core.exceptions

try:
    initial_summary_result, revised_summary_result, critique_text_result = perform_self_reflection(gemini_model, long_text, reflection_requirements)
except google.api_core.exceptions.TooManyRequests as e:
    print("\n[ERROR] 429 Resource Exhausted / Quota Exceeded.")
    print("You have exceeded your Gemini API free-tier rate limits or quota.")
    print("Please wait a minute before retrying, or check your API usage limits at https://aistudio.google.com/")
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")


[ERROR] 429 Resource Exhausted / Quota Exceeded.
You have exceeded your Gemini API free-tier rate limits or quota.
Please wait a minute before retrying, or check your API usage limits at https://aistudio.google.com/
